## Importar Librerías

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
from scipy.stats import chi2_contingency

In [37]:
# Mostrar la versión de TensorFlow
print("TensorFlow version:", tf.__version__)

# Detectar GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPU(s) detectada(s):")
    for gpu in gpus:
        print(" -", gpu)
else:
    print("No se detectaron GPUs.")


TensorFlow version: 2.16.1
GPU(s) detectada(s):
 - PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


## Cargar datos

In [38]:
df = pd.read_csv('data/accidentes.csv')
df.head()

,Nº Expediente,Fecha,Localización,Número,Distrito,Tipo_accidente,Estado_meteorológico,Tipo_vehiculo,Tipo_persona,Rango_edad,Sexo,Lesividad,Gravedad
0,2018S017842,2019-02-04 09:10:00,"CALL. ALBERTO AGUILERA, 1",1,CENTRO,Colisión lateral,Despejado,Motocicleta > 125cc,Conductor,De 45 a 49 años,Hombre,Asistencia sanitaria sólo en el lugar del acci...,Leve
1,2018S017842,2019-02-04 09:10:00,"CALL. ALBERTO AGUILERA, 1",1,CENTRO,Colisión lateral,Despejado,Turismo,Conductor,De 30 a 34 años,Mujer,Asistencia sanitaria sólo en el lugar del acci...,Leve
2,2019S000001,2019-01-01 03:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,CARABANCHEL,Alcance,Se desconoce,Furgoneta,Conductor,De 40 a 44 años,Hombre,Se desconoce,Desconocida
3,2019S000001,2019-01-01 03:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,CARABANCHEL,Alcance,Se desconoce,Turismo,Conductor,De 40 a 44 años,Mujer,Se desconoce,Desconocida
4,2019S000001,2019-01-01 03:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,CARABANCHEL,Alcance,Se desconoce,Turismo,Conductor,De 45 a 49 años,Mujer,Se desconoce,Desconocida


## Análisis de Datos

In [39]:
# 1) Vista general y tipos de dato
print('=== Vista general ===')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
display(df.head(3))

print('\n=== Tipos de dato ===')
display(df.dtypes.value_counts().to_frame('n_columnas'))

=== Vista general ===
Filas: 51,811 | Columnas: 13


,Nº Expediente,Fecha,Localización,Número,Distrito,Tipo_accidente,Estado_meteorológico,Tipo_vehiculo,Tipo_persona,Rango_edad,Sexo,Lesividad,Gravedad
0,2018S017842,2019-02-04 09:10:00,"CALL. ALBERTO AGUILERA, 1",1,CENTRO,Colisión lateral,Despejado,Motocicleta > 125cc,Conductor,De 45 a 49 años,Hombre,Asistencia sanitaria sólo en el lugar del acci...,Leve
1,2018S017842,2019-02-04 09:10:00,"CALL. ALBERTO AGUILERA, 1",1,CENTRO,Colisión lateral,Despejado,Turismo,Conductor,De 30 a 34 años,Mujer,Asistencia sanitaria sólo en el lugar del acci...,Leve
2,2019S000001,2019-01-01 03:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,CARABANCHEL,Alcance,Se desconoce,Furgoneta,Conductor,De 40 a 44 años,Hombre,Se desconoce,Desconocida



=== Tipos de dato ===


,n_columnas
str,13


### Valores nulos

In [40]:
missing = pd.DataFrame({
    'n_nulos': df.isna().sum(),
    'pct_nulos': (df.isna().mean() * 100).round(2)
}).sort_values('pct_nulos', ascending=False)

print('=== Nulos por columna ===')
display(missing)

=== Nulos por columna ===


,n_nulos,pct_nulos
Nº Expediente,0,0.0
Fecha,0,0.0
Localización,0,0.0
Número,0,0.0
Distrito,0,0.0
Tipo_accidente,0,0.0
Estado_meteorológico,0,0.0
Tipo_vehiculo,0,0.0
Tipo_persona,0,0.0
Rango_edad,0,0.0


### Cardinalidad y candidatas a eliminación

In [41]:
card = pd.DataFrame({
    'n_unicos': df.nunique(dropna=False),
    'pct_unicos': (df.nunique(dropna=False) / len(df) * 100).round(2),
    'tipo': df.dtypes.astype(str)
}).sort_values('pct_unicos', ascending=False)

display(card)

# Reglas simples para sugerir columnas a eliminar:
cols_muchos_nulos = missing.query('pct_nulos > 40').index.tolist()
cols_constantes = card.query('n_unicos <= 1').index.tolist()
cols_posible_id = card.query("tipo == 'object' and pct_unicos > 90").index.tolist()

print('\n=== Candidatas a eliminar ===')
print(f'- >40% nulos: {cols_muchos_nulos if cols_muchos_nulos else "ninguna"}')
print(f'- Constantes (1 único valor): {cols_constantes if cols_constantes else "ninguna"}')
print(f'- Posibles ID (object con >90% únicos): {cols_posible_id if cols_posible_id else "ninguna"}')

,n_unicos,pct_unicos,tipo
Nº Expediente,21934,42.33,str
Fecha,19448,37.54,str
Localización,15853,30.60,str
Número,823,1.59,str
Tipo_vehiculo,30,0.06,str
Distrito,21,0.04,str
Tipo_accidente,13,0.03,str
Rango_edad,18,0.03,str
Lesividad,9,0.02,str
Estado_meteorológico,7,0.01,str



=== Candidatas a eliminar ===
- >40% nulos: ninguna
- Constantes (1 único valor): ninguna
- Posibles ID (object con >90% únicos): ninguna


### Distribuciones por tipo de accidente

In [42]:
total_accidentes = df.shape[0]
total_accidentes_por_tipo = df["Tipo_accidente"].value_counts()
print("\n=== Total de accidentes por tipo ===")
display(
    total_accidentes_por_tipo.to_frame("n_accidentes").assign(
        pct_accidentes=lambda x: (x["n_accidentes"] / total_accidentes * 100).round(2)
    )
)


=== Total de accidentes por tipo ===


,n_accidentes,pct_accidentes
Tipo_accidente,,
Colisión fronto-lateral,12905,24.91
Alcance,12220,23.59
Colisión lateral,7511,14.50
Choque contra obstáculo fijo,5920,11.43
Colisión múltiple,3757,7.25
Atropello a persona,3576,6.90
Caída,3006,5.80
Otro,1486,2.87
Colisión frontal,1050,2.03


### Gravedad y Lesividad de los accidentes

In [43]:
print(df['Gravedad'].value_counts(dropna=False))

Gravedad
Desconocida    21769
Ileso          16609
Leve           12860
Grave            539
Fallecido         34
Name: count, dtype: int64


In [44]:
print(df['Lesividad'].value_counts(dropna=False))

Lesividad
Se desconoce                                                 21771
Sin asistencia sanitaria                                     16609
Asistencia sanitaria sólo en el lugar del accidente           7113
Ingreso inferior o igual a 24 horas                           2157
Asistencia sanitaria inmediata en centro de salud o mutua     1551
Atención en urgencias sin posterior ingreso                   1322
Asistencia sanitaria ambulatoria con posterioridad             715
Ingreso superior a 24 horas                                    539
Fallecido 24 horas                                              34
Name: count, dtype: int64


### Eliminar columnas no relevantes y redundates para el modelado

In [45]:
cols_a_eliminar = ['Nº Expediente', 'Fecha', 'Número', 'Localización', 'Gravedad']

cols_presentes = [c for c in cols_a_eliminar if c in df.columns]
if cols_presentes:
    print(f"\nEliminando columnas: {', '.join(cols_presentes)}")
    df = df.drop(columns=cols_presentes)

df.info()


Eliminando columnas: Nº Expediente, Fecha, Número, Localización, Gravedad
<class 'pandas.DataFrame'>
RangeIndex: 51811 entries, 0 to 51810
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Distrito              51811 non-null  str  
 1   Tipo_accidente        51811 non-null  str  
 2   Estado_meteorológico  51811 non-null  str  
 3   Tipo_vehiculo         51811 non-null  str  
 4   Tipo_persona          51811 non-null  str  
 5   Rango_edad            51811 non-null  str  
 6   Sexo                  51811 non-null  str  
 7   Lesividad             51811 non-null  str  
dtypes: str(8)
memory usage: 3.2 MB


## Codificación de variables

In [54]:
# Separamos la variable objetivo del resto de variables predictoras
X = df.drop(columns=['Lesividad']).copy()
y = df['Lesividad'].copy()

onehot_cols = [
    'Distrito',
    'Estado_meteorológico',
    'Tipo_accidente',
    'Sexo'
 ]

embedding_cols = [
    'Rango_edad',
    'Tipo_persona',
    'Tipo_vehiculo'
 ]

# One-hot para columnas nominales
X_encoded = pd.get_dummies(
    X,
    columns=onehot_cols,
    drop_first=True
).copy()

# Label encoding para columnas que irán a Embedding
embedding_encoders = {}
for col in embedding_cols:
    enc = LabelEncoder()
    X_encoded[col] = enc.fit_transform(X[col].astype(str))
    embedding_encoders[col] = enc

# Codificación de la variable objetivo
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y.astype(str))

X_encoded.head()

,Tipo_vehiculo,Tipo_persona,Rango_edad,Distrito_BARAJAS,Distrito_CARABANCHEL,Distrito_CENTRO,Distrito_CHAMARTÍN,Distrito_CHAMBERÍ,Distrito_CIUDAD LINEAL,Distrito_FUENCARRAL-EL PARDO,...,Tipo_accidente_Colisión frontal,Tipo_accidente_Colisión fronto-lateral,Tipo_accidente_Colisión lateral,Tipo_accidente_Colisión múltiple,Tipo_accidente_Despeñamiento,Tipo_accidente_Otro,Tipo_accidente_Solo salida de la vía,Tipo_accidente_Vuelco,Sexo_Hombre,Sexo_Mujer
0,16,0,8,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,False,True,False
1,27,0,5,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,False,False,True
2,13,0,7,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3,27,0,7,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,27,0,8,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


## Modelado

In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Clases objetivo: {len(np.unique(y_encoded))}")

X_train: (41448, 43) | X_test: (10363, 43)
Clases objetivo: 9


In [56]:
# Preparamos los inputs para embeddings + one-hot
input_rango_edad = tf.keras.Input(shape=(1,), name='Rango_edad')
input_tipo_persona = tf.keras.Input(shape=(1,), name='Tipo_persona')
input_tipo_vehiculo = tf.keras.Input(shape=(1,), name='Tipo_vehiculo')

# input_dim = número de categorías codificadas (+1 opcional para OOV; aquí usamos tamaño exacto)
embedding_rango_edad = tf.keras.layers.Embedding(
    input_dim=X_encoded['Rango_edad'].max() + 1, output_dim=4, name='embedding_rango_edad'
 )(input_rango_edad)
embedding_tipo_persona = tf.keras.layers.Embedding(
    input_dim=X_encoded['Tipo_persona'].max() + 1, output_dim=4, name='embedding_tipo_persona'
 )(input_tipo_persona)
embedding_tipo_vehiculo = tf.keras.layers.Embedding(
    input_dim=X_encoded['Tipo_vehiculo'].max() + 1, output_dim=4, name='embedding_tipo_vehiculo'
 )(input_tipo_vehiculo)

onehot_feature_cols = [c for c in X_encoded.columns if c not in embedding_cols]
input_onehot = tf.keras.Input(shape=(len(onehot_feature_cols),), name='onehot_input')

# Concatenamos embeddings y one-hot
concat = tf.keras.layers.Concatenate()([
    tf.keras.layers.Flatten()(embedding_rango_edad),
    tf.keras.layers.Flatten()(embedding_tipo_persona),
    tf.keras.layers.Flatten()(embedding_tipo_vehiculo),
    input_onehot
])

# Diccionarios de entrada listos para model.fit(...)
X_train_inputs = {
    'Rango_edad': X_train['Rango_edad'].astype('int32').values,
    'Tipo_persona': X_train['Tipo_persona'].astype('int32').values,
    'Tipo_vehiculo': X_train['Tipo_vehiculo'].astype('int32').values,
    'onehot_input': X_train[onehot_feature_cols].astype('float32').values,
}
X_test_inputs = {
    'Rango_edad': X_test['Rango_edad'].astype('int32').values,
    'Tipo_persona': X_test['Tipo_persona'].astype('int32').values,
    'Tipo_vehiculo': X_test['Tipo_vehiculo'].astype('int32').values,
    'onehot_input': X_test[onehot_feature_cols].astype('float32').values,
}

print('Inputs preparados:')
for k, v in X_train_inputs.items():
    print(f'- {k}: {v.shape}')

Inputs preparados:
- Rango_edad: (41448,)
- Tipo_persona: (41448,)
- Tipo_vehiculo: (41448,)
- onehot_input: (41448, 40)
